# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 3: K-Means & DBSCAN
**Estimated time: 1 hour**

### Learning Objectives
By the end of this exercise, you will be able to:
- Understand the fundamentals of **Unsupervised Learning** and when to use it
- Apply **K-Means** clustering and tune the number of clusters using the **Elbow Method** and **Silhouette Score**
- Understand and apply **DBSCAN** for density-based clustering
- Compare the strengths and limitations of each algorithm

In this exercise, you will explore **unsupervised learning** techniques covered in Lecture 3. You will cluster a real-world dataset using two different algorithms and learn when each one shines.

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

---
## Setup

🏃 **RUN** the cell below to import libraries.

In [ ]:
# Import needed libraries
# Author: Luca Gudi (lgg.digi@cbs.dk)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Clustering algorithms
from sklearn.cluster import KMeans, DBSCAN

# Evaluation metrics
from sklearn.metrics import silhouette_score

# Display options
plt.rcParams['figure.figsize'] = (10, 6)

---
## 1. Examine the Data

📖 **READ**: In this session we'll be working with the **Seeds dataset**. It contains measurements of seeds from three varieties of wheat (Kama, Rosa, Canadian — 70 each).

The columns are:
- `area` — area of the seed (A)
- `perimeter` — perimeter of the seed (P)
- `compactness` — C = 4 × π × A / P²
- `length` — length of the kernel
- `width` — width of the kernel
- `asymmetry` — asymmetry coefficient
- `groove_length` — length of kernel groove
- `variety` — wheat variety (Kama, Rosa, Canadian)

🏃 **RUN** the cell below to load the data.

In [ ]:
# Load the Seeds dataset
df = pd.read_csv("https://raw.githubusercontent.com/nick-edu/dmmldl/master/Seeds.csv")
print(f"Dataset shape: {df.shape}")
df.head(7)

✏️ **TODO**: Examine the relationships between variables using a **scatter matrix**.

*Hint: Use `scatter_matrix` from `pandas.plotting`.*

In [ ]:
# TODO: Import scatter_matrix and plot it for the dataframe
# from pandas.plotting import ...
# scatter_matrix(...)


---
## 2. K-Means Clustering

📖 **READ**:
K-Means is one of the simplest and most popular clustering algorithms. It works by:
1. Placing **k centroids** randomly
2. **Assigning** each data point to the nearest centroid
3. **Updating** centroids to the mean of their assigned points
4. Repeating steps 2–3 until centroids **stop moving** (convergence)

It is **fast and scalable**, but assumes **spherical, similarly-sized clusters** and requires you to **specify k** in advance.

### 2.1 Basic K-Means

✏️ **TODO**: Perform K-Means clustering with **3 clusters** on the columns `width` and `groove_length`.

*Hints:*
- *Select the two columns into a variable `X`*
- *Use `KMeans(n_clusters=3, random_state=1)`*
- *Call `.fit(X)` to run the algorithm*

In [ ]:
# TODO: Select the columns "width" and "groove_length" into X
# X = ...

# TODO: Create a KMeans model with 3 clusters and fit it
# km = ...
# km.fit(...)


✏️ **TODO**: Inspect the cluster labels that K-Means assigned to each data point.

*Hints:*
- *The labels are stored in `km.labels_`*
- *Use `np.unique(..., return_counts=True)` to see how many points are in each cluster*

In [ ]:
# TODO: Print the cluster labels
# print(km.labels_)

# TODO: Print the count of points in each cluster
# np.unique(...)


✏️ **TODO**: Visualize the clustering result. Plot the data points colored by cluster, and mark the centroids.

*Hints:*
- *Use `plt.scatter(X["width"], X["groove_length"], c=km.labels_, cmap='Set1')`*
- *Centroids are in `km.cluster_centers_` — plot them with a different marker (e.g., `marker='X'`)*
- *Print `km.inertia_` to see the Within-Cluster Sum of Squares (WCSS)*

In [ ]:
# TODO: Plot the clustered data with centroids


# TODO: Print the inertia (WCSS)
# print(f"Inertia (WCSS): {km.inertia_:.2f}")


### ✏️ TODO: Answer the following questions

**Q1: What do the centroid markers represent? What is "Inertia (WCSS)" and why do we want it to be low?**

Your answer: ___

### 2.2 K-Means Helper Function

✏️ **TODO**: Create a reusable function `KmeansAndPlot(col1, col2, nClusters=3)` that:
1. Combines `col1` and `col2` into a dataframe using `pd.concat([col1, col2], axis=1)`
2. Fits K-Means with `nClusters` clusters
3. Plots the data colored by cluster label

This will save you from repeating code every time you want to try different variable combinations.

In [ ]:
# TODO: Define the helper function
def KmeansAndPlot(col1, col2, nClusters=3):
    """
    Performs k-means clustering and plots the resulting clusters.

    Arguments:
    col1, col2 -- columns to cluster and plot
    nClusters -- number of clusters to look for
    """
    # TODO: Combine columns into X
    
    # TODO: Fit KMeans
    
    # TODO: Plot with plt.scatter, coloring by labels
    
    pass


✏️ **TODO**: Use your function to cluster `width` and `groove_length`.

In [ ]:
# TODO: Call KmeansAndPlot with width and groove_length


✏️ **TODO**: Now try clustering with `asymmetry` and `groove_length` instead.

In [ ]:
# TODO: Call KmeansAndPlot with asymmetry and groove_length


### ✏️ TODO: Answer

**Q2: Do the clusters look different when using `asymmetry` and `groove_length` compared to `width` and `groove_length`? Why might that be?**

Your answer: ___

---
## 3. Finding the Optimal k: Elbow Method & Silhouette Score

📖 **READ**:
How do we know the **right number of clusters**? Two common methods:

**Elbow Method:**
- Plot the **inertia** (WCSS) for different values of k
- The "elbow" point — where adding more clusters gives **diminishing returns** — suggests the optimal k

**Silhouette Score:**
- Measures how similar a point is to its own cluster vs. neighboring clusters
- Formula: **(b − a) / max(a, b)** where **a** = mean distance to same cluster, **b** = mean distance to nearest other cluster
- Ranges from **−1** (wrong cluster) to **+1** (well-clustered)
- Often **preferred** over the Elbow Method because it gives a clearer answer

✏️ **TODO**: Write a function `plot_elbow_silhouette(X, k_min=2, k_max=10)` that:
1. Loops over k values from `k_min` to `k_max`
2. For each k: fits K-Means, stores `km.inertia_` and `silhouette_score(X, labels)`
3. Plots both curves side by side using `plt.subplots(1, 2, ...)`

Then call it on `X = df[['width', 'groove_length']]`.

*Hints:*
- *`silhouette_score` is already imported*
- *Use `km.fit_predict(X)` to get labels in one step*

In [ ]:
# TODO: Define the function
def plot_elbow_silhouette(X, k_min=2, k_max=10):
    """
    Plots the Elbow (inertia) curve and Silhouette scores
    for K-Means over a range of cluster counts.
    """
    # TODO: Create lists to store inertia and silhouette scores
    
    # TODO: Loop over k values, fit KMeans, store metrics
    
    # TODO: Create side-by-side plots
    
    pass

# TODO: Call the function on width & groove_length
# X = df[['width', 'groove_length']]
# plot_elbow_silhouette(X)


### ✏️ TODO: Answer

**Q3: At which value of k does the "elbow" appear? Does the Silhouette Score agree? Does this make sense given the dataset?**

Your answer: ___

---
## 4. DBSCAN — Density-Based Clustering

📖 **READ**:
**DBSCAN** identifies clusters as regions of **high density** separated by regions of **low density**. Unlike K-Means, it:
- Does **NOT** require specifying the number of clusters
- Can find clusters of **arbitrary shape**
- Naturally detects **outliers/noise**

Key parameters:
- **`eps` (ε)**: Maximum radius of the neighborhood
- **`min_samples`**: Minimum number of points in the ε-neighborhood to be a **core point**

Point types:
- **Core point**: Has ≥ `min_samples` points within `eps` distance
- **Border point**: Within `eps` of a core point, but fewer than `min_samples` neighbors
- **Noise point**: Neither core nor border — an **outlier** (labelled as `-1`)

### 4.1 DBSCAN Helper Function

✏️ **TODO**: Implement a function `DbscanAndPlot(col1, col2, minSamples=5, eps=0.5)` that:
1. Combines `col1` and `col2` into a dataframe
2. Fits DBSCAN with the given parameters
3. Plots the data colored by cluster label
4. Shows the number of clusters and noise points in the title

*Hints:*
- *Use `DBSCAN(min_samples=..., eps=...)`*
- *Noise points are labelled `-1`*
- *Number of clusters: `len(set(labels)) - (1 if -1 in labels else 0)`*

In [ ]:
# TODO: Define the DBSCAN helper function
def DbscanAndPlot(col1, col2, minSamples=5, eps=0.5):
    """
    Performs DBSCAN clustering and plots the resulting clusters.

    Arguments:
    col1, col2 -- columns to cluster and plot
    minSamples -- minimum samples in neighborhood
    eps -- neighborhood radius
    """
    # TODO: Combine columns, fit DBSCAN, get labels
    
    # TODO: Count clusters and noise points
    
    # TODO: Plot with title showing parameters and results
    
    pass


### 4.2 Applying DBSCAN

✏️ **TODO**: Use your function to perform DBSCAN on `length` and `groove_length`.

In [ ]:
# TODO: DBSCAN on length and groove_length


### 4.3 Effect of the `eps` Parameter

📖 **READ**: Choosing the right `eps` is crucial:
- **Too small** → Most points become noise
- **Too large** → Separate clusters merge into one

✏️ **TODO**: Now call your function again with `eps=0.05`. Compare the results.

In [ ]:
# TODO: Try eps = 0.05 — what changes?


### ✏️ TODO: Answer

**Q4: How does changing `eps` from 0.5 to 0.05 affect the results? Why?**

Your answer: ___

### 4.4 DBSCAN — Manual Application

✏️ **TODO**: Apply DBSCAN manually (without the helper function) to `length` and `groove_length`:
1. Combine the columns with `pd.concat`
2. Instantiate `DBSCAN(min_samples=5, eps=0.5)`
3. Call `.fit(X)` and get the labels from `dbs.labels_`
4. Print the labels and count the clusters/noise

In [ ]:
# TODO: Instantiate DBSCAN and perform clustering
# TODO: Also obtain labels for the resulting clusters

# X = pd.concat([df.length, df.groove_length], axis=1)

# Instantiate DBSCAN

# Perform clustering

# Obtain and print labels


### ✏️ TODO: Answer

**Q5: What are the pros and cons of DBSCAN compared to K-Means?**

| | K-Means | DBSCAN |
| :--- | :--- | :--- |
| **Pros** | ___ | ___ |
| **Cons** | ___ | ___ |

---
## Summary

In this lab, you learned how to:

| Section | Technique | Python Code / sklearn Class |
| :--- | :--- | :--- |
| **2.1 K-Means** | Basic K-Means clustering | `KMeans(n_clusters=k)` |
| **2.2 Helper Function** | Reusable clustering + plotting | Custom function |
| **3. Elbow & Silhouette** | Finding optimal k | `kmeans.inertia_`, `silhouette_score(X, labels)` |
| **4.1 DBSCAN** | Density-based clustering | `DBSCAN(eps=0.5, min_samples=5)` |
| **4.3 eps Tuning** | Effect of eps parameter | `DBSCAN(eps=...)` |

**Key takeaways:**
- **K-Means**: Fast and simple, but needs k in advance and works best on spherical clusters
- **DBSCAN**: Finds arbitrary shapes and handles noise, but sensitive to eps and min_samples
- Use the **Elbow Method** and **Silhouette Score** to find the optimal number of clusters
- Always consider the **shape of your data** when choosing a clustering algorithm